# 16 — One answer key, unchanged predictions (DR-26)

**Question:** How do our saved candidates compare when each event has one agreed answer?

We use DR-55's 56 accepted development answers and exclude one unresolved event. The pilot and later reviews overlap: each event appears only once. We do not retrain, change prompts, tune thresholds or call providers. The original files remain unchanged.

These are repeatedly reviewed development examples, selected partly because they were difficult. They are not a new blind test or a random estimate of production accuracy. the reviewer is the only reviewer. Some accepted answers retain evidence limitations.

Run all cells with the project kernel. Private input reports are required; no API keys or live settings are read. The notebook saves a private comparison and an aggregate-only Markdown summary and graph.


**Private inputs:** Full execution needs local reports excluded from Git. Read `docs/results/reviewed-outcome.md` for public aggregate results. Obtain authorized local inputs to run the analysis; do not commit them.


In [ ]:
import json, sys
from pathlib import Path
from statistics import median
import matplotlib.pyplot as plt
from IPython.display import display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / 'src'))
from notebooks.tools import verify_sha256, show_table, percent
from demo_logic.evaluation import multilabel_metrics
from demo_logic.taxonomy import topic_tags
from demo_logic.notebook_support import save_private_json
INPUTS = {'reports/reference-reconciliation/2026-10-05/approved-reference-v1.json': '4578833534ba147a8cc8157aa880553ef25165484d17720b027ba0afc210dfba', 'reports/gemini-broader/v1/development-results.json': '59849e7e463a30553454c0ff16ed9fc50f0e4e13c6d5d1f134f9de14e008490b', 'reports/gem-adaptation/v1/results.json': '9fdebcabbc4b271d0717a7dce3076846ee94dd776ab82220100de5050c586fea', 'reports/gem-adaptation/v1/qwen-v1/results.json': '06d90f979049d58903f39220eef7aedc9172d9cf1efdf5316206b2768a327d60', 'reports/baselines/20260929T093729499041Z.json': '764cb7d7e188c6802d9a5b7c841d8274f70152487f75251d03de0cc3ba82b5e9', 'reports/classical/20260929T100051642372Z.json': '110b11b9b9d0414780d23550e5a94a4d7f27662ad0629a73f76cd325677a8568', 'reports/semantic/20260929T111146494708Z.json': 'e604bb3b92227f4c302e267828a35a1419eadc4187b3091aba415ea0fd482870', 'reports/context/20260929T114519998841Z.json': 'fed60e022dbf5ada33a4024f00738407e63634bc736b1a4cbbfd76e21a84157b', 'reports/semantic-gbert/20260929T111018666989Z.json': '7d460133e6959281e05af733c87c29691652e02526aa9afa73a170acd043d3df', 'reports/hosted/20260930T150128230609Z.json': '71f6901aa905db8e40621039c81989df1f27fe2e690c1f5dabc113f44da55ea0', 'reports/final-benchmark/v1/results.json': '9625d93dff3d912e7d429e2c642db1030b0d84472fdd7eacbb43a632c4ae814c', 'reports/gemini-broader/v1/followup-results.json': '272f0b6af08f958f74e734ae297211d909c0ddb71871b8576fb1f15e353541ca', 'data/dataset.jsonl': '81ba27a52fb25a7fc1ae0a256522a9c22c39c97dfeac1b8cbbecd2e956d67ca4', 'docs/tagging-rubric.md': '0127f65668271a6bc32ec6de32eb68c146228696ba6ecb69fbaf1a46cdba70bd'}
missing = [p for p in [*INPUTS, "reports/error-analysis/2026-10-05/approved-reference-v1.json"] if not (root / p).is_file()]
if missing:
    raise FileNotFoundError("Private inputs missing: " + ", ".join(missing) + ". Read docs/results/reviewed-outcome.md for public results.")
for path, expected in INPUTS.items():
    verify_sha256(root / path, expected)

def read(path):
    """Read a pinned local report; no network access or inference is performed."""
    return json.loads((root / path).read_text())

reference = read('reports/reference-reconciliation/2026-10-05/approved-reference-v1.json')
assert reference['score_ready'] and reference['version'] == 'dr55-reviewed-v1'
records = {r['external_id']: r for r in reference['records']}
assert len(records) == len(reference['records']) == 57
ids = sorted(i for i, r in records.items() if r['score_eligible'])
assert len(ids) == 56
truth = {i: records[i]['reference_tags'] for i in ids}
snapshot = {r['external_id']: r for r in map(json.loads, (root / 'data/dataset.jsonl').read_text().splitlines())}
historical = {i: snapshot[i]['tag_slugs'] for i in ids}
# Earliest recorded review for each ID: a single consistent before-review comparator.
earlier = {i: records[i]['provenance'][0]['tags'] for i in ids}
tags = sorted(topic_tags('v2'))
show_table(['Reference fact', 'Count'], [
    ['Unique reviewed events', 57], ['Accepted answers', len(ids)], ['Unresolved, excluded', 1],
    ['Answers changed since earliest recorded review', sum(set(earlier[i]) != set(truth[i]) for i in ids)],
])


## 1. Load the saved candidates

Keep each model **and its configuration** separate. Qwen's older prompt, simple rubric prompt and adapted Gem are different candidates. Gemini's Gem search-on and search-off results form a paired search comparison. Comparing Gem with the simple prompt also changes event context and instructions, so it cannot isolate a search effect.

Older fitted models use their already selected cutoff and top-two-plus-exact-ties policy. We preserve those outputs; the new three-tag policy belongs to DR-60.


In [ ]:
candidates = {}

def add(name, rows, source, policy, costs=None):
    """Register saved responses by unique event ID, preserving failed/empty answers.

    Missing IDs are unavailable, not failures. Recorded unsuccessful responses
    remain in the denominator and must not contain fabricated predictions.
    """
    if isinstance(rows, dict):
        rows = [dict(row, external_id=i) for i, row in rows.items()]
    mapping = {x['external_id']: x for x in rows}
    assert len(mapping) == len(rows), 'Duplicate prediction IDs'
    for x in rows:
        assert isinstance(x['predicted'], list)
        assert set(x['predicted']) <= set(tags)
        assert len(x['predicted']) == len(set(x['predicted']))
        if x.get('status', 'ok') != 'ok':
            assert not x['predicted'], 'Failed responses must not become successful tags'
    candidates[name] = dict(rows=mapping, source=source, policy=policy, costs=costs)

broad_path = 'reports/gemini-broader/v1/development-results.json'
broad = read(broad_path)
assert set(ids) <= set(broad['contract']['ids'])
for model, rows in broad['answers'].items():
    add(model + ' | simple rubric', rows, broad_path, 'strict two',
        {x['external_id']: x for x in broad['per_event_costs'] if x['model'] == model})
gem_path = 'reports/gem-adaptation/v1/results.json'
gem = read(gem_path)
for condition, rows in gem['answers'].items():
    add('gemini-3.8-flash | Gem ' + condition, rows, gem_path, 'strict two',
        {x['external_id']: x for x in gem['costs'] if x['condition'] == condition})
qgem_path = 'reports/gem-adaptation/v1/qwen-v1/results.json'
qgem = read(qgem_path)
add(qgem['contract']['model'] + ' | Gem search_off', qgem['answers'], qgem_path, 'strict two')

legacy = [
    ('Production regex', 'reports/baselines/20260929T093729499041Z.json', 'runs', 'candidate', 'keywords'),
    ('Word TF-IDF + logistic regression', 'reports/classical/20260929T100051642372Z.json', 'policies', 'model', 'word'),
    ('Word + character TF-IDF + logistic regression', 'reports/classical/20260929T100051642372Z.json', 'policies', 'model', 'word_plus_char'),
    ('TF-IDF F3 context', 'reports/context/20260929T114519998841Z.json', 'results', 'model', 'F3'),
    ('Multilingual NLI', 'reports/semantic/20260929T111146494708Z.json', 'policies', None, None),
    ('German GBERT NLI', 'reports/semantic-gbert/20260929T111018666989Z.json', 'policies', None, None),
]
for name, path, collection, key, value in legacy:
    report = read(path)
    selected = [p for p in report[collection] if p['policy'] == 'include_cutoff_ties' and (key is None or p[key] == value)]
    assert len(selected) == 1
    add(name, selected[0]['predictions'], path,
        'top two + exact ties; cutoff ' + str(selected[0].get('threshold', 'rule-based')))
old_path = 'reports/hosted/20260930T150128230609Z.json'
old = read(old_path)
for provider, rows in old['check_runs'].items():
    add(old['candidates'][provider]['model'] + ' | Notebook 07 ' + old['selected_prompts'][provider],
        rows, old_path, 'strict two; check predictions only')
show_table(['Candidate', 'Available / 56', 'Saved policy'],
           [[name, f"{len(set(ids) & set(c['rows']))}/56", c['policy']] for name, c in candidates.items()])


## 2. Compare on identical events

The main table includes only candidates with saved predictions for all 56 accepted events. A second table uses the intersection available for **every** candidate. Do not compare a row from one table with a row from the other.

Precision asks how many suggested tags match the answers. Recall asks how many reference tags are found. F1 balances the two. Coverage asks how many events receive at least one suggestion. Our targets remain 80% precision, 80% recall and 95% coverage. A development result reaching them does not establish readiness for deployment.

**Summary:** twelve configurations cover the full 56 events. Adding the two older hosted configurations leaves 32 shared events. The smaller table is a coverage constraint, not a newly selected best-performing subset.


In [ ]:
def score(name, selected_ids, answers):
    """Score aligned IDs, including empty outputs, without modifying predictions."""
    return multilabel_metrics([answers[i] for i in selected_ids],
        [candidates[name]['rows'][i]['predicted'] for i in selected_ids], tags)

full_names = [n for n, c in candidates.items() if set(ids) <= set(c['rows'])]
common_ids = sorted(set(ids).intersection(*(set(c['rows']) for c in candidates.values())))
assert common_ids
cohorts = {'All accepted events': (ids, full_names), 'All-candidate intersection': (common_ids, list(candidates))}
results = {}
for cohort, (selected_ids, names) in cohorts.items():
    results[cohort] = {name: {kind: score(name, selected_ids, answer_key)
        for kind, answer_key in [('historical', historical), ('earliest_review', earlier), ('reviewed', truth)]}
        for name in names}
    print(cohort, '—', len(selected_ids), 'events')
    show_table(['Candidate', 'Precision', 'Recall', 'F1', 'Coverage'],
        [[name, *[percent(values['reviewed'][k]) for k in ['micro_precision','micro_recall','micro_f1','coverage']]]
         for name, values in results[cohort].items()])
for cohort, models in results.items():
    for name, values in models.items():
        assert len({v['coverage'] for v in values.values()}) == 1, 'Reference edits cannot change coverage'


## 3. What changed when the answers changed?

The chart holds the model outputs and events fixed. It compares production labels, the earliest saved review for each event, and the reconciled DR-55 answers. The earliest-review column uses the first recorded answer per ID; it is not a recreation of an old pooled score with duplicate events.

An increase means better agreement with the revised answers. It does not prove the model improved or that all earlier labels were wrong. Definitions, missing context and reviewer judgment also contribute.

**Result:** none of the full-set candidates reaches all three targets. Simple-rubric Qwen finds more correct tags and covers every event; Gemini 3.8 has higher precision but misses more reference tags and abstains more often. Their F1 scores are close.


In [ ]:
names = full_names
fig, ax = plt.subplots(figsize=(11, 7))
for offset, kind, label, color in [(-0.25, 'historical', 'Production labels', '#aeb9c4'),
                                  (0, 'earliest_review', 'Earliest review', '#e6b45d'),
                                  (0.25, 'reviewed', 'DR-55 answers', '#138b84')]:
    ax.barh([i + offset for i in range(len(names))],
            [100 * results['All accepted events'][n][kind]['micro_f1'] for n in names],
            height=0.24, label=label, color=color)
ax.set(yticks=range(len(names)), yticklabels=names, xlim=(0, 100), xlabel='Micro F1 (%)',
       title='Same 56 events and predictions; different answer keys')
ax.legend(loc='lower right', fontsize=8)
ax.tick_params(axis='y', labelsize=8)
fig.tight_layout()
public_dir = root / 'docs/results'
public_dir.mkdir(exist_ok=True)
fig.savefig(public_dir / 'reference-comparison.svg', metadata={'Date': None})
display(fig)
plt.close(fig)
show_table(['Candidate', 'Historical F1', 'Reviewed F1', 'Difference (percentage points)'],
    [[n, percent(v['historical']['micro_f1']), percent(v['reviewed']['micro_f1']),
      round(100 * (v['reviewed']['micro_f1'] - v['historical']['micro_f1']), 1)]
     for n, v in results['All accepted events'].items()])


## 4. Inspect tags and disagreements

Small categories can move sharply after one answer changes. The table shows each tag's support alongside precision and recall for the two practical candidates. The private report retains per-tag results for every candidate.

The private disagreement list separates **recorded reference changes**, **reference/context limitations**, and **remaining prediction disagreements**. These flags may overlap. A disagreement alone does not establish a model mistake; we do not automatically relabel it or ask for another full review.


In [ ]:
shortlist = ['Qwen/Qwen3.6-35B-A3B-FP8 | simple rubric', 'gemini-3.8-flash | simple rubric']
show_table(['Tag', 'Support', 'Qwen precision', 'Qwen recall', 'Gemini precision', 'Gemini recall'],
    [[tag, results['All accepted events'][shortlist[0]]['reviewed']['per_tag'][tag]['support'],
      *[percent(results['All accepted events'][name]['reviewed']['per_tag'][tag][k])
        for name in shortlist for k in ['precision','recall']]] for tag in tags])
disagreements = []
for name in full_names:
    for i in ids:
        predicted = set(candidates[name]['rows'][i]['predicted'])
        expected = set(truth[i])
        if predicted != expected:
            disagreements.append(dict(candidate=name, external_id=i,
                missing=sorted(expected-predicted), extra=sorted(predicted-expected),
                reference_changed=set(earlier[i]) != expected,
                reference_limitation=records[i]['limitation'],
                assessment='Unadjudicated prediction disagreement; do not infer a confirmed model error.'))
# A disclosed, bounded sensitivity check, not a claim that all other labels are certain.
exception_ids = [i for i in ids if records[i].get('review_case') in {3, 5}]
sensitivity_ids = sorted(set(ids)-set(exception_ids))
sensitivity = {n: score(n, sensitivity_ids, truth) for n in full_names}
print('Sensitivity: exclude only DR-55 cases 3 and 5; other recorded limitations remain.')
show_table(['Candidate', 'N', 'Precision', 'Recall', 'F1', 'Coverage'],
    [[n, len(sensitivity_ids), *[percent(v[k]) for k in ['micro_precision','micro_recall','micro_f1','coverage']]]
     for n, v in sensitivity.items()])


## 5. Original waiting time, failures and cost

These are saved **original request durations**, not the time to replay this notebook. The table uses the same 56 events where possible. For legacy local models, per-event request timing and compute cost were not recorded in this format; unavailable is not zero.

USD figures are conservative request estimates, including saved failed-attempt allowances where present. They are not invoices. Free-account Qwen has zero API charges according to the confirmed account setup; local compute was not priced. The last user-reported Google account total was **€1.85**, including **€0.76 on 3 October**, before the later continuation. This account observation cannot be assigned to these 56 events or added to their USD estimates. DR-24 still needs the final accounting reconciliation.

**Result:** simple Qwen took about 1.25 seconds per event at the median; simple Gemini 3.8 took 3.74 seconds. Gem search-on took 8.57 seconds and had two unsuccessful saved responses. Keeping those empty responses in the scoring denominator measures the complete workflow.


In [ ]:
operations = {}
for name in full_names:
    candidate = candidates[name]
    rows = [candidate['rows'][i] for i in ids]
    seconds = [x.get('duration_seconds') for x in rows]
    costs = candidate['costs']
    dollar = sum(costs[i]['usd_upper'] for i in ids) if costs is not None else None
    if name.endswith('| Gem search_off') and name.startswith('Qwen/'):
        dollar = 0 if qgem['summary']['free_account_confirmed'] else None
    operations[name] = dict(
        events=len(ids), failures=sum(x['status'] != 'ok' for x in rows) if all('status' in x for x in rows) else None,
        median_original_seconds=median(seconds) if all(x is not None for x in seconds) else None,
        max_original_seconds=max(seconds) if all(x is not None for x in seconds) else None,
        total_usd_upper=dollar, mean_usd_upper=dollar/len(ids) if dollar is not None else None,
        cost_basis='Saved conservative estimate / confirmed free account; not invoice' if dollar is not None else 'Not measured in this report',
        failure_basis='Saved response status; local prediction rows do not audit infrastructure failures',
    )
def value(x):
    return 'Not recorded' if x is None else round(x, 4)
show_table(['Candidate', 'Response failures', 'Median seconds', 'Max seconds', 'Mean USD upper/event'],
    [[n, value(v['failures']), value(v['median_original_seconds']), value(v['max_original_seconds']), value(v['mean_usd_upper'])]
     for n, v in operations.items()])


## 6. Keep the earlier final benchmark separate

The table below reproduces saved final-benchmark results. Its answers and predictions are unchanged by DR-55. The later Gemini comparison used a benchmark whose results had already been examined; it is a follow-up, not a new untouched test. Do not combine these scores with the development tables.


In [ ]:
original_final = read('reports/final-benchmark/v1/results.json')
followup = read('reports/gemini-broader/v1/followup-results.json')
show_table(['Evaluation', 'Candidate', 'Precision', 'Recall', 'F1', 'Coverage'],
    [[label, name, *[percent(v[k]) for k in ['micro_precision','micro_recall','micro_f1','coverage']]]
     for label, metrics in [('Original final benchmark', original_final['primary_metrics']),
                            ('Previously examined benchmark: later follow-up', followup['metrics'])]
     for name, v in metrics.items()])


## 7. Save the comparison and decide the next step

Read the generated summary for the numbers and tradeoffs. The main practical comparison remains Qwen3.6 versus Gemini3.8 with the simple rubric. Search and Gem variants are separate configurations. Do not choose a model based on F1 alone: recall, coverage, review effort and operating cost still matter.

**Review:** check the cohort counts, read the graph and per-tag table, and confirm the conclusion describes development evidence honestly. No CSV labeling is requested. DR-60 tests the third-tag rule next; DR-24 completes cost reconciliation. DR-27 makes the service decision after those results and the practical review trial.

**Conclusion:** the Gem variants do not establish a clear winner over the simpler prompts. Search-on has slightly higher precision but lower recall, F1 and coverage than Gem search-off. Keep the practical shortlist unchanged; test the three-tag rule separately. These results do not justify automatic tagging yet.


In [ ]:
private_dir = root / 'reports/reconciled-comparison/v2'
report = dict(version='dr26-v2', reference_version=reference['version'], input_sha256=INPUTS,
    ids=ids, common_ids=common_ids, results=results, operations=operations,
    sensitivity=dict(excluded_ids=exception_ids, ids=sensitivity_ids, metrics=sensitivity),
    candidate_sources={n: dict(source=c['source'], policy=c['policy']) for n,c in candidates.items()},
    predictions={n: {i:c['rows'][i]['predicted'] for i in ids if i in c['rows']} for n,c in candidates.items()},
    disagreements=disagreements, api_calls=0, final_test_rescored=False,
    limitation='Repeatedly reviewed development sample, one reviewer, retained evidence exceptions; not a blind test.',
    cost_reconciliation='DR-24 remains pending; USD upper estimates are not EUR invoices')
save_private_json(private_dir / 'results.json', report)
lines = ['# DR-26 — Consistent reviewed comparison', '',
    '5 October 2026. Saved predictions rescored against `dr55-reviewed-v1`. No new API calls.', '',
    '## What was compared', '',
    '56 unique development events have accepted answers; one unresolved event is excluded. Earlier answers and results remain intact. This is a repeatedly reviewed, partly targeted sample with one reviewer, not a blind test.', '',
    '## Same 56 events', '',
    '| Candidate | Precision | Recall | F1 | Coverage |', '|---|---:|---:|---:|---:|']
for n,v in results['All accepted events'].items():
    lines.append('| '+n.replace('|', '—')+' | '+' | '.join(percent(v['reviewed'][k]) for k in ['micro_precision','micro_recall','micro_f1','coverage'])+' |')
lines += ['', '![Effect of changing the answer key](reference-comparison.svg)', '',
    'The chart changes only the answer key. Coverage stays the same. Gains are changes in measured agreement, not improvements to the models.', '',
    '## All-candidate intersection', '', f'All candidates share {len(common_ids)} accepted events. These scores must not be ranked against the 56-event table.', '',
    '| Candidate | Precision | Recall | F1 | Coverage |', '|---|---:|---:|---:|---:|']
for n,v in results['All-candidate intersection'].items():
    lines.append('| '+n.replace('|', '—')+' | '+' | '.join(percent(v['reviewed'][k]) for k in ['micro_precision','micro_recall','micro_f1','coverage'])+' |')
lines += ['', '## What this means', '']
for n in shortlist:
    v=results['All accepted events'][n]['reviewed']
    passes=v['micro_precision']>=.8 and v['micro_recall']>=.8 and v['coverage']>=.95
    lines.append(f"- **{n}:** {'meets' if passes else 'does not meet'} all three quality targets on these development answers.")
a=results['All accepted events']['gemini-3.8-flash | Gem search_off']['reviewed']
b=results['All accepted events']['gemini-3.8-flash | Gem search_on']['reviewed']
lines += [f"- Within the adapted Gemini Gem, search changes F1 by {100*(b['micro_f1']-a['micro_f1']):+.1f} percentage points and coverage by {100*(b['coverage']-a['coverage']):+.1f} points. This is the paired search comparison; it does not isolate context or prompt effects.",
    '- Two newly reviewed politics assignments retain evidence/rubric caveats. Notebook 16 shows the same comparison excluding only those two cases; other reference limitations remain. Do not interpret that sensitivity check as a verified clean dataset.',
    '- Per-tag support and errors are available in Notebook 16. Rare tags and missing context still limit interpretation. Differences from references are not automatically confirmed model mistakes.', '',
    '## Operations on the same 56 events', '',
    '| Candidate | Response failures | Median original seconds | Max original seconds | Mean USD upper/event |', '|---|---:|---:|---:|---:|']
for n,v in operations.items():
    lines.append('| '+n.replace('|', '—')+' | '+str(value(v['failures']))+' | '+' | '.join(str(value(v[k])) for k in ['median_original_seconds','max_original_seconds','mean_usd_upper'])+' |')
lines += ['', 'Saved request durations are not cache replay times. Local computation was not priced. USD estimates are conservative, not invoices. The last reported Google total (€1.85, including €0.76 on 3 October before later continuation) is an account observation, not a cost for this subset. DR-24 still needs final reconciliation.', '',
    '## Next steps', '',
    'Review Notebook 16 and this summary. Keep Qwen3.6 and Gemini3.8 simple-rubric configurations as the practical comparison. Complete DR-24 accounting, then DR-60’s separate third-tag experiment and DR-58’s human-effort trial before DR-27’s service decision. Do not lower the targets based on this selected sample.', '',
    '## Reproduce', '',
    'Run `notebooks/16_reconciled_reference_comparison.ipynb` with the private pinned inputs. It verifies input fingerprints, saves `reports/reconciled-comparison/v2/results.json`, and regenerates this aggregate summary and SVG. The original final benchmark and later follow-up remain separately labeled in section 6; neither is changed here.', '']
(public_dir / 'reviewed-comparison.md').write_text('\n'.join(lines))
print('Saved private results and aggregate summary. No model calls or reference edits.')


## 8. DR-61 follow-up: did the review change the comparison?

The ten-case review is complete. Four accepted answers changed; two additional cases are unclear. This leaves 54 accepted events from the original 57. Case 8 keeps social-issues + work as a reviewer interpretation supported by background reporting, not a verified event match. Other recorded context limitations remain.

Compare the old and new answers on the **same retained IDs**. Do not compare 56-event and 54-event scores as if only labels changed. All saved candidate configurations are rescored where they share these IDs; the smaller intersection remains separate. No new predictions, threshold changes or rubric edits are made. These are repeatedly reviewed development findings, not a fresh test.


In [ ]:
dr61_path = 'reports/error-analysis/2026-10-05/approved-reference-v1.json'
verify_sha256(root / dr61_path, 'b508d0a3b9b803f7d018816c753e1673860c1a297acd8733607bf6c1f67fe6fe')
dr61 = read(dr61_path)
assert dr61['score_ready'] and dr61['version'] == 'dr61-reviewed-v1'
new_truth = {x['external_id']: x['reference_tags'] for x in dr61['records'] if x['score_eligible']}
assert len(new_truth) == 54 and set(new_truth) <= set(ids)
retained = sorted(new_truth)
retained_common = sorted(set(common_ids) & set(retained))
paired = {}
for label, selected_ids, names in [('All retained', retained, full_names),
                                  ('All-candidate intersection', retained_common, list(candidates))]:
    paired[label] = {n: {'before': score(n, selected_ids, truth),
                         'after': score(n, selected_ids, new_truth)} for n in names}
    print(label, len(selected_ids))
    show_table(['Candidate', 'Before F1', 'After precision', 'After recall', 'After F1', 'Coverage'],
        [[n, percent(v['before']['micro_f1']), *[percent(v['after'][k])
          for k in ['micro_precision','micro_recall','micro_f1','coverage']]] for n,v in paired[label].items()])
    assert all(v['before']['coverage'] == v['after']['coverage'] for v in paired[label].values())
save_private_json(root / 'reports/reconciled-comparison/dr61-v1/results.json',
    dict(version='dr61-paired-v1', reference_sha256='b508d0a3b9b803f7d018816c753e1673860c1a297acd8733607bf6c1f67fe6fe', prior_input_sha256=INPUTS,
         ids=retained, common_ids=retained_common, paired=paired,
         excluded_from_dr55=sorted(set(ids)-set(retained)), api_calls=0,
         note='Same retained IDs before/after; original DR26 metrics preserved.'))
lines = ['# DR-61 — Reviewed outcome', '',
    'Four accepted answers changed. Two additional events are unclear, leaving 54 accepted events and three unresolved events in the 57-event union. No predictions or rubric definitions changed.', '',
    'The table holds the 54 event IDs fixed. Before uses DR-55 answers; after uses DR-61 answers. Coverage cannot change when only reference tags change.', '',
    '| Candidate | Before F1 | After precision | After recall | After F1 | Coverage |',
    '|---|---:|---:|---:|---:|---:|']
for n,v in paired['All retained'].items():
    lines.append('| '+n.replace('|','—')+' | '+percent(v['before']['micro_f1'])+' | '+' | '.join(percent(v['after'][k]) for k in ['micro_precision','micro_recall','micro_f1','coverage'])+' |')
lines += ['', '## What the review teaches us', '',
    '- Reference corrections can add a missing topic or remove an unsupported topic. They need not benefit both models equally.',
    '- One agreement control became unclear. Agreement between the models and the old answer is not proof of correctness.',
    '- Location and background reports help identify context but do not automatically establish the demands of a specific event. One healthcare reference retains a labor interpretation with a date-matching limitation.',
    '- Labels describe subjects, not whether an event supports or opposes them. Opposition to development can still concern housing.',
    '- No general rubric meaning change was approved. Preserve case-specific interpretations and source gaps rather than treating them as new rules.',
    '- The sample is small, targeted and repeatedly reviewed by one person. These results cannot establish production accuracy or erase the models’ own mistakes.', '',
    '## Next step', '',
    'Review section 8 of Notebook 16. No more labeling is requested for DR-61. Keep the practical shortlist unchanged and test the third-tag policy under DR-60, separately from reference corrections. DR-24 cost reconciliation is still required before the service decision.', '',
    'Private source: reports/error-analysis/2026-10-05/approved-reference-v1.json. Private paired metrics: reports/reconciled-comparison/dr61-v1/results.json.']
(public_dir / 'reviewed-outcome.md').write_text('\n'.join(lines)+'\n')
